In [1]:
import os
import sys
import datetime
import pandas as pd
from upbit.client import Upbit
import time
import traceback

# upper_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
upper_dir = "/home/trade_core"
sys.path.append(upper_dir)
from loggers.logger import KimpBotLogger

In [2]:
upbit_access_key = "x2AKLfsRtAgRxFjQ7p9TZTAg6E1SveoqfNfD8MK8"
upbit_secret_key = "svEran52QdsUyJdxAPYoTgFT2MXsHc5ZiKsKPxTL"

In [37]:
class UserUpbitAdaptor:
    def __init__(self, logging_dir=None):
        self.user_client_dict = {}
        self.retry_term_sec = 0.2
        self.retry_count_limit = 2
        self.upbit_plug_logger = KimpBotLogger("user_upbit_plug", logging_dir).logger
        self.upbit_plug_logger.info(f"user_upbit_plug_logger started.")

    def load_user_client(self, access_key, secret_key):
        user_client = self.user_client_dict.get(access_key)
        if user_client is None:
            self.user_client_dict[access_key] = Upbit(access_key, secret_key)
            return self.user_client_dict[access_key]
        else:
            return user_client
        
    def get_spot_balance(self, access_key, secret_key, return_dict=None):
        upbit_client = self.load_user_client(access_key, secret_key)
        result_df = pd.DataFrame(upbit_client.Account.Account_info()['result'])
        result_df.loc[:, ['balance','locked','avg_buy_price']] = result_df[['balance','locked','avg_buy_price']].astype(float)
        result_df = result_df.rename(columns={'currency':'asset', 'balance':'free', 'locked':'used'})
        if return_dict is None:
            return result_df
        else:
            return_dict['res'] = result_df

    def get_balance(self, access_key, secret_key, market_type='SPOT'):
        if market_type == "SPOT":
            return self.get_spot_balance(access_key, secret_key)
        elif market_type == "USD_M":
            raise Exception(f"market_type: {market_type} is not supported yet.")
        elif market_type == "COIN_M":
            raise Exception(f"market_type: {market_type} is not supported yet.")
        else:
            raise Exception(f"Invalid market_type: {market_type}")
    
    def check_api_key(self, access_key, secret_key, futures=False):
        self.user_client_dict.pop(access_key, None)
        client = self.load_user_client(access_key, secret_key)
        if futures:
            raise Exception(f"futures market is not supported yet.")
        try:
            response = client.Account.Account_info()['response']
            if response['status_code'] == 200:
                return (True, 'OK')
            else:
                return (False, response['text'])
        except Exception as e:
            self.user_client_dict.pop(access_key, None)
            return (False, str(e))

In [38]:
user_upbit_adaptor = UserUpbitAdaptor()

2024-02-20 18:18:25,669 - user_upbit_plug - INFO - user_upbit_plug_logger started.
2024-02-20 18:18:25,669 - user_upbit_plug - INFO - user_upbit_plug_logger started.
2024-02-20 18:18:25,669 - user_upbit_plug - INFO - user_upbit_plug_logger started.
2024-02-20 18:18:25,669 - user_upbit_plug - INFO - user_upbit_plug_logger started.
2024-02-20 18:18:25,669 - user_upbit_plug - INFO - user_upbit_plug_logger started.


In [39]:
client = user_upbit_adaptor.load_user_client(upbit_access_key, upbit_access_key)

In [41]:
user_upbit_adaptor.user_client_dict.pop(upbit_access_key, None)

UpbitClient(https://api.upbit.com)

In [42]:
user_upbit_adaptor.check_api_key(upbit_access_key, upbit_access_key)

(False,
 '{"error":{"message":"Failed to verify Jwt token.","name":"jwt_verification"}}')

In [24]:
client.Account.Account_info()

{'remaining_request': {'group': 'default', 'min': '1800', 'sec': '29'},
 'response': {'url': 'https://api.upbit.com/v1/accounts',
  'headers': {'Date': 'Tue, 20 Feb 2024 18:14:45 GMT', 'Content-Type': 'application/json;charset=UTF-8', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Remaining-Req': 'group=default; min=1800; sec=29', 'Vary': 'Origin, Access-Control-Request-Method, Access-Control-Request-Headers', 'Cache-Control': 'no-cache, no-store, max-age=0, must-revalidate'},
  'status_code': 200,
  'reason': 'OK',
  'text': '[{"currency":"KRW","balance":"118487197.04307504","locked":"0","avg_buy_price":"0","avg_buy_price_modified":true,"unit_currency":"KRW"},{"currency":"ETH","balance":"0.00000004","locked":"0","avg_buy_price":"2958528.42763717","avg_buy_price_modified":false,"unit_currency":"KRW"},{"currency":"WAVES","balance":"0.00000001","locked":"0","avg_buy_price":"2521.22000402","avg_buy_price_modified":false,"unit_currency":"KRW"},{"currency":"ARK","balance":"0.0

In [14]:
user_upbit_adaptor.check_api_key(upbit_access_key, upbit_access_key)

{'remaining_request': {'group': 'default', 'min': '1800', 'sec': '29'}, 'response': {'url': 'https://api.upbit.com/v1/accounts', 'headers': {'Date': 'Tue, 20 Feb 2024 18:12:03 GMT', 'Content-Type': 'application/json; charset=utf-8', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Remaining-Req': 'group=default; min=1800; sec=29', 'Vary': 'Origin, Access-Control-Request-Method, Access-Control-Request-Headers'}, 'status_code': 401, 'reason': 'Unauthorized', 'text': '{"error":{"message":"Failed to verify Jwt token.","name":"jwt_verification"}}', 'content': b'{"error":{"message":"Failed to verify Jwt token.","name":"jwt_verification"}}', 'ok': False}, 'result': {'error': {'message': 'Failed to verify Jwt token.', 'name': 'jwt_verification'}}}


(True, 'OK')

In [5]:
balance_df = user_upbit_adaptor.get_spot_balance(upbit_access_key, upbit_secret_key)
balance_df

,asset,free,used,avg_buy_price,avg_buy_price_modified,unit_currency
0,KRW,118487197.043075,0.0,0.0,True,KRW
1,ETH,0.0,0.0,2958528.427637,False,KRW
2,WAVES,0.0,0.0,2521.220004,False,KRW
3,ARK,0.0,0.0,2065.0,False,KRW
4,BCH,0.0,0.0,340470.959448,False,KRW
5,STORJ,0.0,0.0,935.25234,False,KRW
6,USDT,0.058399,0.0,1079.359474,False,KRW
7,WAXP,0.0,0.0,76.552876,False,KRW
8,POLYX,0.0,0.0,287.0,False,KRW
9,HIFI,0.0,0.0,895.940898,False,KRW
